[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yy726/GPQA-eval/blob/submission-yy726/notebooks/13_distill_gkd.ipynb)

# 13 · 蒸馏：让更强的老师教 S1

> ⚠️ **只作参考，不参与排名。** 老师是官方后训练模型；比赛规则（`main` 的 RULES 第 2、3 条）不允许加载其他已后训练的模型、也不允许比赛期间用别的模型生成训练数据。
> 这个 notebook 只放在 `submission-yy726` 分支上做研究对比。

学生默认从 notebook 15 的共享 SFT 检查点 S1 开始（`INIT_MODEL="auto"`），所以结果可以和 notebook 11（GRPO）、12（DPO）直接比较。两种蒸馏方式（`DISTILL_MODE`）：

### `"answers"`（默认）：答案级蒸馏，老师可以是任何模型

老师回答留出的 4 选 1 科学题（题面和 GPQA 一样，来自 MegaScience，S1 没见过解答），**只保留最后选对的回答**，学生在这些回答上做 SFT。
不需要老师和学生同词表，所以可以用比 Qwen3-32B 强得多的模型（按顺序尝试，第一个能加载的就用）：

| 老师 | GPQA-Diamond（模型卡） | 说明 |
|---|---|---|
| `Qwen/Qwen3.8-27B-FP8` | **89.2**（思考模式） | 2026-08 发布；31 GB，A100 40GB 上留给 KV 缓存的显存不多，生成偏慢 |
| `Qwen/Qwen3.5-35B-A3B-GPTQ-Int4` | 84.2 | MoE（每个 token 只激活 3B），24 GB，生成快 |
| `Qwen/Qwen3-30B-A3B-Instruct-2507-FP8` | 70.4（非思考） | 备选 |

（现在的老师 Qwen3-32B：思考模式 68.4、非思考 54.6。Qwen3.5 / 3.8 的词表是 248,320，和 Qwen3 的 151,936 不同，所以它们**不能**做下面的 logits 蒸馏。）
老师用非思考模式作答（推理写在回答里，长度和 S1 的风格接近）；`TEACHER_THINKING="on"` 时让老师先思考、只保留 `</think>` 之后的部分。
老师在单独的进程里跑（vLLM），跑完显存全部释放，再加载学生。

### `"logits"`：GKD，学生对齐老师的逐 token 分布

结构参考 Unsloth 官方 [Qwen3 (0.6B) Knowledge Distillation](https://github.com/unslothai/notebooks/blob/main/python_scripts/Qwen3_(0.6B)-Knowledge_Distillation.py)：
TRL `GKDTrainer` + Unsloth 的分块损失。老师**必须和学生同词表**，所以只能在 Qwen3 这一代里选：默认先试 `Qwen/Qwen3-30B-A3B-Instruct-2507`
（非思考模式 GPQA 70.4，比 Qwen3-32B 的 54.6 高很多；没有现成的 4-bit 版本，要下载 61 GB 再现场量化），加载失败就退回 `unsloth/Qwen3-32B-unsloth-bnb-4bit`。
- `lmbda`：0 = 离线（在数据集给定的答案上对齐老师分布；默认）；1 = 在线（学生自己生成、老师打分；很慢）。`beta`：0 = 正向 KL，1 = 反向 KL，0.5 = JSD。
- 数据：MegaScience 教材题（物理 / 化学 / 生物）+ GPQA 题面的 4 选 1 科学题。
- 结束符位置不算损失：学生以 `<|endoftext|>` 结束，老师在那里想写 `<|im_end|>`（Qwen3 Base 学不会它，X1 的问题）。

**不用守着**：Run all 之后可以离开。老师的回答、adapter、评测结果都存在 Google Drive（开头会弹一次授权），断线后重新 Run all 会跳过已完成的部分；跑完自动断开 runtime。

| 模式 | 老师 → 学生 | 数据 | 时间（估） |
|---|---|---|---|
| `quick` | Qwen3-1.7B → Qwen3-0.6B-Base | 约 150 题 / 250 条 | T4 约 20 分钟 |
| `a100` + `answers` | Qwen3.8-27B → S1 | 3000 题，保留答对的（预计 2500 条左右），2 个 epoch | 老师生成 1–1.5 小时 + 训练 20 分钟 + 评测 |
| `a100` + `logits` | Qwen3-30B-A3B-Instruct-2507（4-bit）→ S1 | 约 6000 条，1 个 epoch | 约 2–2.5 小时 |

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

In [ ]:
# 克隆提交分支（数据准备 / 去污染 / 共用代码 scripts/recipe_utils.py 都在里面）
import os, sys
if not os.path.exists("GPQA-eval"):
    !git clone -q -b submission-yy726 https://github.com/yy726/GPQA-eval.git
if os.path.basename(os.getcwd()) != "GPQA-eval":   # 重跑这一格时不要再进一层
    %cd GPQA-eval
!git pull -q
if os.path.abspath("scripts") not in sys.path:
    sys.path.insert(0, os.path.abspath("scripts"))
import recipe_utils
print("reloaded:", recipe_utils.reload_scripts())   # 改了 scripts/ 之后重跑这一格即可，不用重启 runtime

In [ ]:
# 安装（约 3–5 分钟）。输出不隐藏：如果安装报错请看这里。
!pip install --upgrade -q uv
try: import numpy, PIL; _numpy = f'numpy=={numpy.__version__}'; _pil = f'pillow=={PIL.__version__}'
except: _numpy = "numpy"; _pil = "pillow"
!uv pip install unsloth vllm wandb {_numpy} {_pil}

# GKD：用 Unsloth 官方蒸馏示例测过的 TRL 版本（新版 TRL 把 GKDTrainer 移到了 trl.experimental，Unsloth 的分块损失补丁不一定生效）
!uv pip install --no-deps "trl==0.25.1"

In [ ]:
import os, importlib.metadata as md, torch
os.environ["UNSLOTH_VLLM_STANDBY"] = "1"
# T4（计算能力 7.5）不支持 FlashInfer 采样器（A100 不受影响）
if torch.cuda.get_device_capability()[0] < 8:
    os.environ["UNSLOTH_VLLM_NO_FLASHINFER"] = "1"
    os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"
    print("T4 detected: FlashInfer disabled for vLLM")
GPU_NAME = torch.cuda.get_device_name(0)
GPU_MEM_GB = round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1)
VERSIONS = {}
for p in ["unsloth", "unsloth_zoo", "vllm", "trl", "transformers", "torch", "peft", "wandb"]:
    try: VERSIONS[p] = md.version(p)
    except md.PackageNotFoundError: VERSIONS[p] = None
print(GPU_NAME, GPU_MEM_GB, "GB"); print(VERSIONS)

In [ ]:
# Colab 左侧 🔑 Secrets：HF_TOKEN（必需；先在 https://huggingface.co/datasets/Idavidrein/gpqa 接受条款）
#                      WANDB_API_KEY（可选；没有就不记录 W&B）
for _k in ["HF_TOKEN", "WANDB_API_KEY"]:
    try:
        from google.colab import userdata
        _v = userdata.get(_k)
        if _v:
            os.environ[_k] = _v
    except Exception:
        pass
from huggingface_hub import login, whoami
login(token = os.environ.get("HF_TOKEN"))
HF_USER = whoami(token = os.environ.get("HF_TOKEN"))["name"]
print("HF user:", HF_USER, "|", "W&B key found" if os.environ.get("WANDB_API_KEY") else "No WANDB_API_KEY: W&B logging disabled")

## 0. 配置

In [ ]:
EXP_NAME = "K1"
MODE     = "a100"                 # "quick" | "a100"
DISTILL_MODE = "answers"          # "answers" = 老师作答、学生在答对的回答上 SFT（老师可以是任何模型）| "logits" = GKD（老师必须同词表）
STUDENT  = None                   # None = 用预设；例如 "unsloth/Qwen3-4B-Base"
INIT_MODEL = "auto"               # "auto" = notebook 15 的共享 SFT 检查点 f"{HF_USER}/qwen3-4b-s1-sft"；None = 从 base 开始
PUSH_REPO  = "auto"               # "auto" = f"{HF_USER}/qwen3-4b-<EXP_NAME>-distill"（私有）；None = 只存到持久目录
EVAL_BEFORE = True                # 训练前先测一次学生（独立进程，约 5–10 分钟）
GEN_BUDGET  = None                # 评测的生成预算。None = 预设（a100: 8192，和排行榜一致）；16384 = 只在参考分支放宽
USE_DRIVE   = True                # 老师的回答、adapter、评测结果都存到 Google Drive：runtime 重置后不会丢
RESUME      = True                # 已经完成的部分直接读结果，不重跑；False = 全部重来
AUTO_DISCONNECT = True            # 全部跑完后自动断开 runtime（不再消耗 CU）
CHAT_STYLE  = "olmo"              # 和其它 notebook 一致：回答以 <|endoftext|> 结束

# ---- DISTILL_MODE = "answers"
ANSWER_TEACHERS = None            # None = 用预设的列表（按顺序尝试，第一个能加载的就用）；也可以自己填 ["..."]
TEACHER_THINKING = "off"          # "off" = 老师把推理写在回答里（推荐）；"on" = 老师先思考，只保留 </think> 之后的部分
TEACHER_N   = 3000                # 让老师回答多少道留出的科学题（a100 的数量；quick 自动缩小）
ANSWER_EPOCHS = 2                 # 学生在老师的回答上训练几个 epoch
ANSWER_LR   = 5e-5

# ---- DISTILL_MODE = "logits"（GKD）
TEACHER     = None                # None = 用预设的列表；OOM 时 "unsloth/Qwen3-14B-unsloth-bnb-4bit"
LMBDA       = 0.0                 # 0 离线 / 1 在线
BETA        = 0.5                 # 0 正向 KL / 1 反向 KL
TEMPERATURE = 1.0
# 每项从一个 HF 数据集抽 n 条（a100 的数量；quick 自动乘 0.05）。见 experiments/DATASETS.md
DISTILL_SOURCES = [
    dict(dataset = "MegaScience/MegaScience", n = 4000, include = ["physics", "chemistry", "biology"]),   # 教材题（按 subject 过滤）
    # dict(dataset = "allenai/Dolci-Instruct-SFT-7B", n = 2000, include = ["science"], exclude = ["safety", "wildguard", "python", "code", "aya", "precise", "flan", "logic"]),
]
MCQ_N = 2000                      # GPQA 题面的 4 选 1 科学题 + 分步解答（scripts/science_mix.py，和 S1 的选择题同源）

WANDB_PROJECT = os.environ.get("WANDB_PROJECT", "gpqa-hackathon")
WANDB_ENTITY  = os.environ.get("WANDB_ENTITY")

PRESETS = dict(
    quick = dict(student="unsloth/Qwen3-0.6B-Base", base_model="Qwen/Qwen3-0.6B-Base",
                 answer_teachers=["unsloth/Qwen3-1.7B"], logit_teachers=[("unsloth/Qwen3-1.7B", False)],
                 max_seq_len=1024, scale=0.05, steps=30, lr=2e-4, batch=1, accum=4, lora_rank=16,
                 teacher_max_tokens=512, teacher_gpu_util=0.8, sft_batch=4, sft_accum=2,
                 eval_configs=["gpqa_diamond"], eval_n=60, eval_max_tokens=1024, eval_samples=1, eval_gpu_util=0.5,
                 shuffle_buffer=1000),
    a100 = dict(student="unsloth/Qwen3-4B-Base", base_model="Qwen/Qwen3-4B-Base",
                answer_teachers=["Qwen/Qwen3.8-27B-FP8", "Qwen/Qwen3.5-35B-A3B-GPTQ-Int4", "Qwen/Qwen3-30B-A3B-Instruct-2507-FP8"],
                # (模型, 是否 4-bit 加载)：30B-A3B-Instruct-2507 没有现成的 bnb-4bit，下载 61 GB 后现场量化；失败就退回 32B
                logit_teachers=[("Qwen/Qwen3-30B-A3B-Instruct-2507", True), ("unsloth/Qwen3-32B-unsloth-bnb-4bit", True)],
                max_seq_len=4096, scale=1.0, steps=-1, lr=5e-5, batch=1, accum=8, lora_rank=32,
                teacher_max_tokens=2048, teacher_gpu_util=0.92, sft_batch=4, sft_accum=4,
                eval_configs=["gpqa_diamond", "gpqa_main"], eval_n=None, eval_max_tokens=8192, eval_samples=4,
                eval_gpu_util=0.5, shuffle_buffer=20000),
)
Q = dict(PRESETS[MODE])
if GEN_BUDGET:
    Q.update(eval_max_tokens = GEN_BUDGET)
assert DISTILL_MODE in ("answers", "logits")
STUDENT = STUDENT or Q["student"]
ANSWER_TEACHERS = ANSWER_TEACHERS or Q["answer_teachers"]
LOGIT_TEACHERS = [(TEACHER, "bnb-4bit" in TEACHER)] if TEACHER else Q["logit_teachers"]
BASE_MODEL, MAX_SEQ_LEN = Q["base_model"], Q["max_seq_len"]
from recipe_utils import resolve_init_model, sft_repo_name, persist_dir, StageCache
INIT_MODEL = resolve_init_model(INIT_MODEL, HF_USER, STUDENT)
if PUSH_REPO == "auto":
    PUSH_REPO = sft_repo_name(HF_USER, STUDENT).replace("-s1-sft", f"-{EXP_NAME.lower()}-distill")
PERSIST = persist_dir(f"{EXP_NAME}_{MODE}", USE_DRIVE)     # 开头会弹一次 Drive 授权
CACHE = StageCache(PERSIST, enabled = RESUME)
CACHE.check_config(dict(mode = DISTILL_MODE, student = STUDENT, init_model = INIT_MODEL, chat_style = CHAT_STYLE,
                        answer_teachers = ANSWER_TEACHERS, thinking = TEACHER_THINKING, teacher_n = TEACHER_N,
                        answer_epochs = ANSWER_EPOCHS, answer_lr = ANSWER_LR, logit_teachers = LOGIT_TEACHERS, lmbda = LMBDA,
                        beta = BETA, temperature = TEMPERATURE, sources = DISTILL_SOURCES, mcq_n = MCQ_N,
                        **{k: Q[k] for k in ("max_seq_len", "scale", "steps", "lr", "lora_rank", "eval_n", "eval_max_tokens")}))
print(EXP_NAME, MODE, DISTILL_MODE, "student:", INIT_MODEL or STUDENT, "-> push:", PUSH_REPO)
print("teachers:", ANSWER_TEACHERS if DISTILL_MODE == "answers" else LOGIT_TEACHERS, "| persist dir:", PERSIST, "| done:", CACHE.done())

In [ ]:
import json
from recipe_utils import (TABLE_COLS, run_script, wandb_init, log_eval, print_table, decontam_rows, prepare_sft_sources,
                          config_kwargs, setup_chat, paired_delta, load_guard, restore_lora, end_runtime, RESPONSE_PARTS)

RUN_NAME = f"{EXP_NAME}-distill-{DISTILL_MODE}-{STUDENT.split('/')[-1]}-{MODE}"
wandb, REPORT_TO = wandb_init(WANDB_PROJECT, RUN_NAME, Q | dict(
    exp = EXP_NAME, recipe = "distill", distill_mode = DISTILL_MODE, mode = MODE, student = STUDENT, init_model = INIT_MODEL,
    answer_teachers = ANSWER_TEACHERS, teacher_thinking = TEACHER_THINKING, teacher_n = TEACHER_N, answer_epochs = ANSWER_EPOCHS,
    logit_teachers = LOGIT_TEACHERS, lmbda = LMBDA, beta = BETA, temperature = TEMPERATURE, sources = DISTILL_SOURCES, mcq_n = MCQ_N,
    chat_style = CHAT_STYLE, gpu = GPU_NAME, gpu_mem_gb = GPU_MEM_GB, versions = VERSIONS, reference_only = True), WANDB_ENTITY)
REPORT = dict(notebook = "13_distill_gkd", exp = EXP_NAME, mode = MODE, distill_mode = DISTILL_MODE, student = STUDENT,
              init_model = INIT_MODEL, q = Q, reference_only = True)
RESULTS, _PER_Q = {}, {}

def quick_eval_subprocess(model_path, tag, gpu_util, samples = None):
    # 独立进程跑 vLLM（scripts/quick_eval.py），结束后显存全部释放；只输出聚合数字。结果存在持久目录，断线后不重跑
    out = f"{PERSIST}/eval_{tag}.json"
    if not (RESUME and CACHE.enabled and os.path.exists(out)):
        run_script(["scripts/quick_eval.py", "--model", model_path, "--configs", *Q["eval_configs"],
                    "--max-tokens", Q["eval_max_tokens"], "--max-model-len", Q["eval_max_tokens"] + 4096,
                    "--samples", samples or Q["eval_samples"], "--gpu-util", gpu_util, "--tag", tag, "--out", out]
                   + (["--n", Q["eval_n"]] if Q["eval_n"] else []))
    else:
        print(f"[i] {tag}: 读回已有的评测结果 {out}")
    rep = json.load(open(out))
    RESULTS[tag], _PER_Q[tag] = rep["results"], rep.get("per_question", {})
    for cfg, r in RESULTS[tag].items():
        print(tag, cfg, r)
    log_eval(wandb, RESULTS[tag], tag)

if EVAL_BEFORE:   # 这时还没往 GPU 上加载任何模型；base 不收尾（每个回答写满预算），只采样 1 次
    quick_eval_subprocess(INIT_MODEL or BASE_MODEL, "0_init" if INIT_MODEL else "0_base", 0.85,
                          samples = None if INIT_MODEL else 1)

## 1. 老师作答（只在 `DISTILL_MODE="answers"` 时运行）

`scripts/teacher_answers.py` 在独立进程里用 vLLM 加载老师，回答 `TEACHER_N` 道留出的科学题，只保留最后选对的回答，结尾统一成 `The answer is (X).`。
老师列表按顺序尝试；回答存在持久目录，断线后不会重新生成。**这一格要在加载学生之前跑**（老师要占满显存）。

In [ ]:
TEACHER_INFO = {}
ANSWERS_FILE = f"{PERSIST}/teacher_answers.jsonl"
if DISTILL_MODE == "answers":
    if RESUME and CACHE.has("teacher"):
        TEACHER_INFO = CACHE.load("teacher")
        print("[i] 老师的回答已经生成过：", TEACHER_INFO)
    else:
        if not os.path.exists("data/s1/rl_mcq.jsonl"):
            run_script(["scripts/science_mix.py", "--out-dir", "data/s1", "--pools-only"], show_tail = 20)   # 下载 MegaScience 约 1.9 GB
        _n = max(20, int(TEACHER_N * Q["scale"]))
        for _teacher in ANSWER_TEACHERS:
            try:
                # 从题库的中段取题：notebook 11 的 GRPO 从开头取、notebook 12 的 DPO 从末尾取
                run_script(["scripts/teacher_answers.py", "--teacher", _teacher, "--pool", "data/s1/rl_mcq.jsonl", "--out", ANSWERS_FILE,
                            "--n", _n, "--skip", 1700 if Q["scale"] >= 1 else 0, "--max-tokens", Q["teacher_max_tokens"],
                            "--gpu-util", Q["teacher_gpu_util"], "--thinking", TEACHER_THINKING], show_tail = 15)
                TEACHER_INFO = json.load(open(ANSWERS_FILE.replace(".jsonl", ".stats.json")))
                break
            except Exception as e:
                print(f"[!] 老师 {_teacher} 没跑成（{type(e).__name__}），试下一个")
        assert TEACHER_INFO.get("kept"), "没有一个老师跑成功：检查上面的报错（显存 / vLLM 是否支持这个模型），或改 ANSWER_TEACHERS"
        CACHE.save("teacher", TEACHER_INFO)
    REPORT["teacher"] = TEACHER_INFO
    print("teacher:", TEACHER_INFO.get("teacher"), "| 老师在这些题上的准确率:", TEACHER_INFO.get("teacher_accuracy"),
          "| 保留:", TEACHER_INFO.get("kept"), "/", TEACHER_INFO.get("questions"))

## 2. 加载学生（16-bit LoRA）；`logits` 模式再加载老师（冻结）

In [ ]:
import torch, gc
from unsloth import FastLanguageModel

if DISTILL_MODE == "logits":
    import unsloth
    from unsloth.models.rl_replacements import RL_FUNCTIONS
    _patches = [getattr(f, "__name__", "") for f in RL_FUNCTIONS.get("gkd_trainer", [])]
    assert any("compute_loss" in n for n in _patches), (
        f"这个 Unsloth 没有给 GKD 接上分块损失（{_patches}）：Runtime → Restart session 后重跑（让上面的安装生效）")
    print("unsloth", unsloth.__version__, "| GKD chunked loss wired:", _patches)

# 这一格可以反复运行：学生（和老师）已经加载过就直接复用，LoRA 权重重置
_KEY = (INIT_MODEL or STUDENT, MAX_SEQ_LEN, Q["lora_rank"], DISTILL_MODE, str(LOGIT_TEACHERS) if DISTILL_MODE == "logits" else "")
teacher = globals().get("teacher")
if load_guard(globals(), _KEY, var = "student"):
    student, tokenizer = FastLanguageModel.from_pretrained(
        INIT_MODEL or STUDENT, max_seq_length = MAX_SEQ_LEN, dtype = None, load_in_4bit = False)
    tokenizer = getattr(tokenizer, "tokenizer", tokenizer)
    STOP_IDS = setup_chat(tokenizer, student, style = CHAT_STYLE)   # 回答以 eos=<|endoftext|> 结束（照 OLMo 3）
    head = student.get_output_embeddings()
    assert head.weight.dim() == 2, f"output head is not dense: {type(head.weight).__name__}"
    student = FastLanguageModel.get_peft_model(
        student, r = Q["lora_rank"], lora_alpha = 2 * Q["lora_rank"], lora_dropout = 0, bias = "none",
        target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        use_gradient_checkpointing = "unsloth", random_state = 3407)

    TEACHER_NAME = None
    if DISTILL_MODE == "logits":
        gc.collect(); torch.cuda.empty_cache()
        _free, _total = torch.cuda.mem_get_info(0)
        max_memory = {0: max(int(_free - 2.0 * 2**30), 0)}   # 给老师留预算，保留 2GB 余量
        print(f"free {_free / 2**30:.1f} of {_total / 2**30:.1f} GiB -> teacher budget {max_memory[0] / 2**30:.1f} GiB")
        for _name, _4bit in LOGIT_TEACHERS:                  # 按顺序尝试；同词表是硬要求（下面检查）
            try:
                teacher, teacher_tok = FastLanguageModel.from_pretrained(
                    _name, max_seq_length = MAX_SEQ_LEN, dtype = None, load_in_4bit = _4bit, max_memory = max_memory)
                TEACHER_NAME = _name
                break
            except Exception as e:
                print(f"[!] 老师 {_name} 没加载成（{type(e).__name__}: {str(e)[:200]}），试下一个")
                teacher = None; gc.collect(); torch.cuda.empty_cache()
        assert teacher is not None, "没有一个老师加载成功：OOM 的话设 TEACHER = 'unsloth/Qwen3-14B-unsloth-bnb-4bit'"
        teacher_tok = getattr(teacher_tok, "tokenizer", teacher_tok)
        teacher.eval()
        for p in teacher.parameters():
            p.requires_grad_(False)
        # 检查：同词表（宽度 + 实际 id）、老师和学生确实不同
        s_vocab, t_vocab = student.config.get_text_config().vocab_size, teacher.config.get_text_config().vocab_size
        assert s_vocab == t_vocab, f"vocab width mismatch: student {s_vocab} vs teacher {t_vocab}"
        probe = "The quick brown fox jumps over 0123456789 lazy dogs, naively. <|im_start|>user\nHi<|im_end|>"
        assert tokenizer(probe).input_ids == teacher_tok(probe).input_ids, "tokenizers disagree: GKD cannot compare distributions"
        _ids = tokenizer("The capital of France is", return_tensors = "pt").input_ids.to(student.device)
        with torch.no_grad():
            _gap = (student(input_ids = _ids).logits.float() - teacher(input_ids = _ids).logits.float()).abs().max().item()
        print(f"teacher {TEACHER_NAME} | vocab {s_vocab} ok | max teacher-student logit gap {_gap:.3f}")
    _LOADED_KEY = _KEY
else:
    print("[i] 模型已经加载，直接复用。")
print("fresh LoRA:", restore_lora(student))                  # 重跑时从头开始，而不是接着上一次的权重
print(f"GPU memory after loading: {torch.cuda.memory_allocated() / 2**30:.1f} GiB")
REPORT["logit_teacher"] = globals().get("TEACHER_NAME")

## 3. 蒸馏数据

- `answers`：老师答对的回答（上一节），过 GPQA 13-gram 去污染。
- `logits`：`prepare_sft_sources` 抽教材题、`scripts/science_mix.py` 给出 GPQA 题面的选择题，同样去污染。

整条放不进 `MAX_SEQ_LEN` 的样本会被丢掉。

In [ ]:
from datasets import Dataset
from collections import Counter

if DISTILL_MODE == "answers":
    rows = [dict(json.loads(l), dataset = TEACHER_INFO.get("teacher")) for l in open(ANSWERS_FILE)]
else:
    rows = prepare_sft_sources(DISTILL_SOURCES, STUDENT, MAX_SEQ_LEN, scale = Q["scale"], shuffle_buffer = Q["shuffle_buffer"])
    if MCQ_N > 0:
        _n_mcq = max(1, int(MCQ_N * Q["scale"]))
        run_script(["scripts/science_mix.py", "--out-dir", "data/s1_gkd", "--tokenizer", STUDENT, "--max-tokens", MAX_SEQ_LEN - 64,
                    "--mcq", _n_mcq, "--textbook", "--natural", 0, "--pool-factor", 1], show_tail = 5)   # 下载 MegaScience 约 1.9 GB
        rows += [dict(json.loads(l), dataset = "MegaScience/MegaScience (4-option MCQ, GPQA format)") for l in open("data/s1_gkd/sft_pool.jsonl")]
rows, dropped = decontam_rows(rows, lambda r: {"messages": r["messages"]}, "distill_all")

def _fits(msgs):
    whole = tokenizer.apply_chat_template(msgs, tokenize = True)
    prompt = tokenizer.apply_chat_template(msgs[:-1], tokenize = True, add_generation_prompt = True)
    whole = whole["input_ids"] if hasattr(whole, "keys") else whole
    prompt = prompt["input_ids"] if hasattr(prompt, "keys") else prompt
    return len(whole) <= MAX_SEQ_LEN and len(prompt) < len(whole)

before = len(rows)
rows = [r for r in rows if _fits(r["messages"])]
import random; random.Random(0).shuffle(rows)
train_ds = Dataset.from_list([{"messages": r["messages"]} for r in rows])
DATA = dict(rows = len(rows), dropped_decontam = dropped, dropped_too_long = before - len(rows),
            by_dataset = Counter(r["dataset"] for r in rows).most_common(),
            tokens = sum(r.get("n_tokens") or 0 for r in rows))
REPORT["data"] = DATA
wandb.config.update({"data": DATA}, allow_val_change = True)
print(DATA)

## 4. 训练

- `answers`：普通 SFT（只在回答部分算 loss），`ANSWER_EPOCHS` 个 epoch。
- `logits`：`GKDConfig`（不设 fp16/bf16，Unsloth 按架构自动选精度）。

训练完的 adapter 存在持久目录；断线后重新 Run all 不会重训。

In [ ]:
LORA_DIR = f"{PERSIST}/distill_lora"
if RESUME and CACHE.has("train") and os.path.exists(f"{LORA_DIR}/adapter_model.safetensors"):
    TRAIN = CACHE.load("train")
    print("[i] 训练已经完成过，读回 adapter：", TRAIN, restore_lora(student, LORA_DIR))
else:
    print("fresh LoRA:", restore_lora(student))
    before = {n: p.detach().clone() for n, p in student.named_parameters() if p.requires_grad}
    torch.cuda.reset_peak_memory_stats()
    if DISTILL_MODE == "answers":
        from trl import SFTTrainer, SFTConfig
        from unsloth.chat_templates import train_on_responses_only
        text_ds = Dataset.from_list([{"text": tokenizer.apply_chat_template(r["messages"], tokenize = False)} for r in rows])
        trainer = SFTTrainer(model = student, tokenizer = tokenizer, train_dataset = text_ds, args = SFTConfig(**config_kwargs(
            SFTConfig,
            dataset_text_field = "text", max_length = MAX_SEQ_LEN, per_device_train_batch_size = Q["sft_batch"],
            gradient_accumulation_steps = Q["sft_accum"], warmup_steps = 10, max_steps = Q["steps"],
            num_train_epochs = ANSWER_EPOCHS, learning_rate = ANSWER_LR, logging_steps = 5, optim = "adamw_8bit",
            weight_decay = 0.01, lr_scheduler_type = "cosine", seed = 3407, report_to = REPORT_TO, run_name = RUN_NAME,
            output_dir = "outputs/distill_sft", save_strategy = "no")))
        instr_part, resp_part = RESPONSE_PARTS[CHAT_STYLE]
        trainer = train_on_responses_only(trainer, instruction_part = instr_part, response_part = resp_part)
    else:
        try:
            from trl import GKDConfig, GKDTrainer
        except ImportError:                       # 新版 TRL
            from trl.experimental.gkd import GKDConfig, GKDTrainer
        config = GKDConfig(**config_kwargs(
            GKDConfig,
            output_dir = "outputs/gkd", per_device_train_batch_size = Q["batch"], gradient_accumulation_steps = Q["accum"],
            warmup_steps = 10, max_steps = Q["steps"], num_train_epochs = 1, learning_rate = Q["lr"], logging_steps = 5,
            optim = "adamw_8bit", weight_decay = 0.01, lr_scheduler_type = "linear", seed = 3407,
            report_to = REPORT_TO, run_name = RUN_NAME, save_strategy = "no",
            max_length = MAX_SEQ_LEN, lmbda = LMBDA, beta = BETA, temperature = TEMPERATURE, max_new_tokens = 512))
        trainer = GKDTrainer(model = student, teacher_model = teacher, args = config, train_dataset = train_ds,
                             processing_class = tokenizer)
        assert type(trainer.args).__name__ == "GKDConfig", f"config was downcast to {type(trainer.args).__name__}"
        assert trainer.args.lmbda == LMBDA and trainer.args.beta == BETA
        # 结束符位置不算蒸馏损失：学生以 <|endoftext|> 结束，老师在这里想写 <|im_end|>（Qwen3 Base 学不会它，X1 的问题）
        _EOS_ID, _collate = tokenizer.eos_token_id, trainer.data_collator
        def _collate_without_eos(features):
            batch = _collate(features)
            if "labels" in batch:
                batch["labels"][batch["input_ids"] == _EOS_ID] = -100
            return batch
        try:
            _probe = _collate_without_eos([trainer.train_dataset[0]])
            print("eos positions masked in a sample batch:", int((_probe["input_ids"] == _EOS_ID).sum()),
                  "| labelled tokens:", int((_probe["labels"] != -100).sum()))
            trainer.data_collator = _collate_without_eos
        except Exception as e:
            print("[!] could not wrap the collator; the end-token position stays in the loss:", type(e).__name__, e)

    result = trainer.train()
    changed = sum(1 for n, p in student.named_parameters() if p.requires_grad and not torch.equal(p.detach(), before[n]))
    losses = [h["loss"] for h in trainer.state.log_history if "loss" in h]
    TRAIN = dict(mode = DISTILL_MODE, steps = trainer.state.global_step, minutes = round(result.metrics["train_runtime"] / 60, 1),
                 loss_first = losses[0] if losses else None, loss_last = losses[-1] if losses else None,
                 adapters_changed = f"{changed}/{len(before)}", peak_mem_gb = round(torch.cuda.max_memory_allocated() / 2**30, 1))
    assert changed > 0, "no adapter changed: the student did not learn"
    student.save_pretrained(LORA_DIR)             # adapter 存在持久目录
    CACHE.save("train", TRAIN)
    del trainer, before
REPORT["train"] = TRAIN
print(TRAIN)

## 5. 合并学生 → 推到 HF 私有仓库 → 独立进程评测（先释放老师的显存）

In [ ]:
teacher = None
gc.collect(); torch.cuda.empty_cache()
if not os.path.exists("outputs/merged_student/config.json") or not (RESUME and CACHE.has("merged")):
    student.save_pretrained_merged("outputs/merged_student", tokenizer, save_method = "merged_16bit")
    CACHE.save("merged", {"path": "outputs/merged_student"})
if PUSH_REPO and not CACHE.has("pushed"):
    try:
        try:
            student.push_to_hub_merged(PUSH_REPO, tokenizer, save_method = "merged_16bit", private = True, token = os.environ.get("HF_TOKEN"))
        except TypeError:
            student.push_to_hub_merged(PUSH_REPO, tokenizer, save_method = "merged_16bit", token = os.environ.get("HF_TOKEN"))
        print(f"pushed (private): https://huggingface.co/{PUSH_REPO}")
        CACHE.save("pushed", {"model": PUSH_REPO})
    except Exception as e:
        print("[!] 没有推到 HF（adapter 在持久目录里，不会丢）：", type(e).__name__, str(e)[:200])
quick_eval_subprocess("outputs/merged_student", "1_distill", Q["eval_gpu_util"])

## 6. 汇总（W&B 上也有同样的表；最后打印的 JSON 可以贴回给 Claude）

In [ ]:
# 训练前后的配对比较（同一批题，后 − 前，单位：分；|z| < 2 = 看不出差别）
COMPARE = {}
_tags = list(_PER_Q)
for _a, _b in zip(_tags, _tags[1:]):
    COMPARE[f"{_a} -> {_b}"] = {cfg: paired_delta(_PER_Q[_a][cfg], _PER_Q[_b][cfg])
                                for cfg in _PER_Q[_b] if cfg in _PER_Q[_a]}
print("compare:", COMPARE)

rows = print_table(RESULTS)
REPORT.update(quick_eval = RESULTS, compare = COMPARE, versions = VERSIONS, gpu = GPU_NAME, gpu_mem_gb = GPU_MEM_GB,
              wandb = wandb.run.url if REPORT_TO == "wandb" else None)
os.makedirs("outputs", exist_ok = True)
json.dump(REPORT, open(f"outputs/report_{EXP_NAME}.json", "w"), indent = 1, default = str)
if REPORT_TO == "wandb":
    wandb.log({"gpqa_quick_eval": wandb.Table(columns = TABLE_COLS, data = rows)})
    art = wandb.Artifact(f"report-{RUN_NAME}", type = "report")
    art.add_file(f"outputs/report_{EXP_NAME}.json")
    wandb.log_artifact(art)
    wandb.finish()
if "PERSIST" in globals():   # 断开之后也能在 Drive 里找到
    json.dump(REPORT, open(f"{PERSIST}/report_{EXP_NAME}.json", "w"), indent = 1, default = str)
print("=" * 30, "把下面整段 JSON 贴回给 Claude", "=" * 30)
print(json.dumps(REPORT, ensure_ascii = False, default = str))

## 7. 断开 runtime

`AUTO_DISCONNECT=True` 时这一格在 60 秒后释放 runtime，不再消耗 CU。

In [ ]:
end_runtime(AUTO_DISCONNECT)